# Example use-case: LISA UCB convolution of double white-dwarf binaries at current-day

This example fleshes out the steps required to estimate the population of observable double white dwarf systems in the LISA band. Relevant studies are: https://arxiv.org/abs/2405.20484





Several ingredients are necessary here:
- population-synthesis results that contain white-dwarfs
- a Milky-Way galaxy star formation rate history model
- a method to evolve 

Convolution-by-sampling was developed especially for this project, as we want to 'generate' double white dwarf systems at a certain lookback time, and evolve them (through gravitational-wave radiation) to the present day.

The convolution broadly is done as follows:
- In a given lookback-time bin we calculate the total mass formed into stars
- We use that to generate double white dwarf systems (using mass_formed * yield-per-mass-formed) 
- We assign a birth time to these systems (with values bound by the edges of the lookback time bin)
- We 'evolve' these systems up to the current day under the influence of gravitational-wave radiation. We make use of Legwork ([Wagg et al 2021](https://ui.adsabs.harvard.edu/abs/2022ApJS..260...52W/abstract)) in this example.
- Filter out certain systems (those that would interact, those that would merge, etc)
- Calculate detection probabilities for the rest based on their position (either randomly assigned or motivated by a spatially-defined SFH) in the Milkyway and their system properties.
- Use this information to predict observable populations of DWD systems

In the following piece of code I show how we do this.

In [ ]:
"""
Functions to convolve the T0 format with sampling

TODO: move the calculations to the post-convolution hook
TODO: determine which systems that are (at present day) in the lisa frequency range should have interacted through RLOF
TODO: of the systems that are not RLOFing and are within the lisa waveband, store: indices, source.f_orb_now. the rest can be retrieved elsewhere
"""

import os
import json
import time
import copy
import astropy.units as u
import legwork as lw
import numpy as np
import astropy.constants as const
import pandas as pd
import h5py

from syntheticstellarpopconvolve import convolve, default_convolution_config
from syntheticstellarpopconvolve.general_functions import temp_dir

from mass_normalisation import get_mass_norm
from functions import get_period, is_rlofing
from DrawPositionsSeparable import sample_distances_simple
from syntheticstellarpopconvolve.convolve_stochastically import (
    select_dict_entries_with_new_indices,
)

TMP_DIR = temp_dir("code", "convolve_stochastically", clean_path=True)


def post_convolution_function(
    config, job_dict, sfr_dict, data_dict, result_dict, convolution_instruction
):
    """
    Post-convolution function to handle integrating the systems forward in time and finding those that end up in the LISA waveband.

    using local_indices to select everything and using Alexey's distance sampler to handle sampling the distances
    """

    # unpack data
    system_indices = result_dict["indices"]
    event_lookback_times = result_dict["event_lookback_times"]
    local_indices = np.arange(len(system_indices))

    # select system properties
    sma = data_dict["semimajor_axis"][system_indices]
    m_1 = data_dict["mass1"][system_indices]
    m_2 = data_dict["mass2"][system_indices]
    eccentricity = data_dict["eccentricity"][system_indices]
    periods = get_period(sma, m_1, m_2)
    f_orb_i = (1 / periods).to(u.Hz)

    # sample distances
    dist = sample_distances_simple(NBin=len(system_indices), age=age)

    #########
    # Set up legwork sources
    sources = lw.source.Source(
        m_1=m_1,
        m_2=m_2,
        ecc=eccentricity,
        f_orb=f_orb_i,
        dist=dist,
        interpolate_g=len(local_indices) > 1000,
    )

    ##########
    # TODO: use the below steps to improve the selection of systems in band
    # ask; how long until separation = RLOF separatin
    # then: how long does it take until with current separation to go to that seperation
    # then compare that time to
    # check if the distance matches a frequency thats within the waveband.

    #########
    # Evolve the systems until today
    t_evol = event_lookback_times

    sources.evolve_sources(t_evol)

    f_orb_now = sources.f_orb

    ####
    # categorisations
    lower_bound_LISA_passband = 1e-5 * u.Hz
    upper_bound_LISA_passband = 1e-1 * u.Hz

    # 1) doesnt enter lisa waveband today. so also nt in the past (maybe near future)
    # 2) are currently in lisa band. maybe also in the past (but fro which point)
    # 3) are merged now. but they ahve been in lisa band in the past (and from which point)
    # 4) for both 2 and 3, we should filter out the 'interacting' systems

    ##############
    # determine (un)merged systems
    local_indices_merged_systems = local_indices[f_orb_now >= 1e2 * u.Hz]
    local_indices_unmerged_systems = local_indices[f_orb_now < 1e2 * u.Hz]
    config["logger"].warning(
        f"Of the total of {len(local_indices)} systems {len(local_indices_merged_systems)} are merged by today and {len(local_indices_unmerged_systems)} are not"
    )

    f_orb_now_unmerged_systems = f_orb_now[f_orb_now < 1e2 * u.Hz]

    ##############
    # determine unmerged systems in LISA passband
    query_unmerged_systems_within_LISA_passband = (
        f_orb_now_unmerged_systems >= lower_bound_LISA_passband
    ) & (f_orb_now_unmerged_systems <= upper_bound_LISA_passband)

    #
    local_indices_unmerged_systems_within_LISA_passband = (
        local_indices_unmerged_systems[query_unmerged_systems_within_LISA_passband]
    )

    print(
        f"Of the {len(local_indices_unmerged_systems)} unmerged systems {len(local_indices_unmerged_systems_within_LISA_passband)} are within the lisa frequency passband ([{lower_bound_LISA_passband},{upper_bound_LISA_passband}])"
    )

    # return only data from now unmerged systems within the lisa passband

    result_dict = select_dict_entries_with_new_indices(
        sampled_data_dict=result_dict,
        new_indices=local_indices_unmerged_systems_within_LISA_passband,
    )

    # add dists
    result_dict["dists"] = (
        dist[local_indices_unmerged_systems_within_LISA_passband] * u.kpc
    )

    return result_dict


LIGHTWEIGHT = False

###################
# Read T0 output
start = time.time()

if LIGHTWEIGHT:
    BinCodex_events_filename = (
        "/home/david/Desktop/bincodex_results/example_BinCodex.h5"
    )
else:
    BinCodex_events_filename = "/home/david/Desktop/bincodex_results/Seba_BinCodex.h5"

#
BinCodex_T0_events = pd.read_hdf(
    BinCodex_events_filename,
    "T0",
)

##################
# update T0 output

# get mass normalisation
mass_normalisation_fiducial = get_mass_norm(IC_model="fiducial", binary_fraction=0.5)

# set normalised yield
BinCodex_T0_events["normalized_yield"] = 1 / mass_normalisation_fiducial

# Query the dataset to select the formation of the WDs

# to check if things start with some number its easier to turn them into strings
BinCodex_T0_events["str_event"] = BinCodex_T0_events["event"].astype(str)
BinCodex_T0_events["str_type1"] = BinCodex_T0_events["type1"].astype(str)
BinCodex_T0_events["str_type2"] = BinCodex_T0_events["type2"].astype(str)

# first, lets query the type-changing events. Any type-change will do
wd_binaries = BinCodex_T0_events.query("str_event.str.startswith('1')")

# The type should change to a WD-type (and the other should already be one)
wd_binaries = wd_binaries.query("str_type1.str.startswith('2')")
wd_binaries = wd_binaries.query("str_type2.str.startswith('2')")

# to be sure lets only select the first ones that remain for each system
# Drop duplicates based on 'system_id', keeping only the first occurrence
# wd_binaries = wd_binaries.drop_duplicates(subset='UID', keep='first')

# lets delete the string versions of the columns again
wd_binaries = wd_binaries.drop(columns=["str_event", "str_type1", "str_type2"])

# lets also delete the original dataframe
del BinCodex_T0_events

stop = time.time()
print("created queried dataframe")
print("took {}".format(stop - start))

##################
#

# create file
input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# add group for events
input_hdf5_file.create_group("input_data/events")

# Write population config to file
input_hdf5_file.create_dataset("config/population", data=json.dumps({}))

# close
input_hdf5_file.close()

# store the data frame in the hdf5file
wd_binaries.to_hdf(input_hdf5_filename, key="input_data/events/stochastic_example")

#
convolution_config = copy.copy(default_convolution_config)
convolution_config["input_filename"] = input_hdf5_filename
convolution_config["output_filename"] = output_hdf5_filename
convolution_config["tmp_dir"] = TMP_DIR
convolution_config["redshift_interpolator_data_output_filename"] = os.path.join(
    TMP_DIR, "interpolator_dict.p"
)
convolution_config["multiply_by_time_binsize"] = False

###
# convolution instructions
convolution_config["convolution_instructions"] = [
    {
        "input_data_type": "event",
        "convolution_type": "sample",
        "input_data_name": "stochastic_example",
        "output_data_name": "stochastic_example",
        "ignore_metallicity": True,
        "post_convolution_function": post_convolution_function,
        "data_column_dict": {
            # required
            "normalized_yield": "normalized_yield",
            "delay_time": {"column_name": "time", "unit": u.Myr},
        },
    },
]

#
convolution_config["time_type"] = "lookback_time"
# convolution_config["convolution_lookback_time_bin_edges"] = np.arange(0, 4, 0.5) * u.Gyr

# construct the sfr-dict (NOTE: this uses absolute SFR, not metallicity dependent)
sfr_dict = {}
if LIGHTWEIGHT:
    sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 1) * u.Gyr).to(u.yr)
else:
    sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 0.25) * u.Gyr).to(u.yr)

sfr_dict["starformation_rate_array"] = (
    1e-6 * np.ones(sfr_dict["lookback_time_bin_edges"].shape[0] - 1) * u.Msun / u.yr
)  # example of a constant star-formation rate. this could be anything of course.

# store
convolution_config["SFR_info"] = sfr_dict

input_hdf5_file = h5py.File(input_hdf5_filename, "r")

# convolve
convolve(config=convolution_config)

print("finished convolution")


# read out content and integrate until today
with h5py.File(convolution_config["output_filename"], "r") as output_hdf5_file:
    print(
        output_hdf5_file[
            "output_data/event/stochastic_example/stochastic_example/convolution_results"
        ].keys()
    )

    formation_time_bin_keys = list(
        output_hdf5_file[
            "output_data/event/stochastic_example/stochastic_example/convolution_results"
        ].keys()
    )

    ################
    #
    total_in_waveband_lisa = 0

    # loop over the formation-time bins
    formation_time_bin_keys = sorted(
        formation_time_bin_keys, key=lambda x: float(x.split(" ")[0])
    )
    for formation_time_bin_key in formation_time_bin_keys:

        # formation_time_bin_key = "3500000000.0 yr"
        print("=================================")
        print(f"formation_time_bin_key: {formation_time_bin_key}")
        print("=================================")

        ###########
        # Read out data

        # convert units
        unit_dict = json.loads(
            output_hdf5_file[
                f"output_data/event/stochastic_example/stochastic_example/convolution_results/{formation_time_bin_key}"
            ].attrs["units"]
        )
        unit_dict = {key: u.Unit(val) for key, val in unit_dict.items()}
        print(unit_dict)

This example can be made more sophisticated by e.g.:
- Using a spatially-defined star-formation rate history. One can provide a list of starformation histories to the code, each element then representing a part of the grid where the SFR is defined in.
- Splitting 